# VibeFit AI Agent + RAG — Colab Demo

Notebook تعليمي يوضح كيف يقرر **Agent** محتوى رسائل البريد الذكية في VibeFit.

**ملاحظة:** هذا Notebook للعرض والتعلم — لا يرسل Gmail فعليًا.

## ما هو Agent؟

- **Automation:** ينفّذ خطوات ثابتة (إذا X ثم Y).
- **Agent:** يقرر **أي أدوات** يستخدم و**ما المحتوى** بناءً على السياق.

## ما هي Tools؟

دوال Python (أو RPCs) يستدعيها Agent: جلب التقييم، المتابعات، RAG، إلخ.

## ما هو RAG?

Retrieval-Augmented Generation — جلب مقاطع معرفة من Supabase ثم تضمينها في prompt.

## لماذا n8n + Agent?

- **n8n** ينظم الجدولة، الجلب، Gmail، وتسجيل الأحداث.
- **Agent** يقرر: هل نرسل؟ ماذا نقول؟ هل نحتاج RAG؟

In [ ]:
# Install dependencies
!pip -q install openai supabase pandas pydantic python-dotenv

In [ ]:
import os
import json
from typing import Any, Optional
from pydantic import BaseModel, Field
from openai import OpenAI
from supabase import create_client

# Colab Secrets — never hardcode keys in notebook source
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    SUPABASE_URL = userdata.get('SUPABASE_URL')
    SUPABASE_KEY = userdata.get('SUPABASE_KEY')
except Exception:
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    SUPABASE_URL = os.getenv('SUPABASE_URL')
    SUPABASE_KEY = os.getenv('SUPABASE_KEY')

assert OPENAI_API_KEY and SUPABASE_URL and SUPABASE_KEY, 'Set Colab secrets first'

client = OpenAI(api_key=OPENAI_API_KEY)
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('Clients ready')

In [ ]:
class ProactiveEmail(BaseModel):
    should_send: bool = True
    event_type: str
    subject: str = ''
    greeting: str = ''
    body: str = ''
    recommended_actions: list[str] = Field(default_factory=list)
    cta_text: str = ''
    cta_url: str = ''
    footer_notice: Optional[str] = None
    used_personal_data: bool = False
    used_rag: bool = False
    reason_not_sent: Optional[str] = None
    selected_tools: list[str] = Field(default_factory=list)

In [ ]:
def retrieve_knowledge(query: str, category: str | None = None, limit: int = 3):
    payload = {'search_query': query, 'result_limit': limit, 'min_score': 0.03}
    if category:
        payload['category_filter'] = category
    res = supabase.rpc('search_knowledge_documents', payload).execute()
    return res.data or []

def get_user_assessment(user_id: str):
    res = supabase.table('assessments').select('*').eq('user_id', user_id).order('created_at', desc=True).limit(1).execute()
    return (res.data or [None])[0]

def get_latest_recommendation(user_id: str):
    res = supabase.table('recommendations').select('*').eq('user_id', user_id).order('created_at', desc=True).limit(1).execute()
    return (res.data or [None])[0]

def get_weekly_checkins(user_id: str, limit: int = 8):
    res = supabase.table('weekly_checkins').select('*').eq('user_id', user_id).order('week_start', desc=True).limit(limit).execute()
    return res.data or []

def calculate_progress_summary(checkins: list[dict]) -> dict[str, Any]:
    if not checkins:
        return {'weeks': 0, 'avg_adherence_pct': None, 'avg_energy': None}
    adh = [round((c.get('adherence_rate', 0) if c.get('adherence_rate', 0) <= 1 else c.get('adherence_rate', 0)/100)*100) for c in checkins]
    energy = [c.get('energy_level') for c in checkins if c.get('energy_level') is not None]
    return {
        'weeks': len(checkins),
        'avg_adherence_pct': round(sum(adh)/len(adh), 1) if adh else None,
        'avg_energy': round(sum(energy)/len(energy), 1) if energy else None,
        'latest': checkins[0],
    }

In [ ]:
EVENT_TOOL_PLAN = {
    'user_signed_up': ['profile_context'],
    'assessment_completed': ['get_user_assessment'],
    'recommendation_completed': ['get_user_assessment', 'get_latest_recommendation'],
    'adherence_dropped': ['get_weekly_checkins', 'calculate_progress_summary', 'retrieve_knowledge'],
    'low_energy_detected': ['get_weekly_checkins', 'retrieve_knowledge'],
    'weekly_summary_due': ['get_weekly_checkins', 'calculate_progress_summary', 'get_latest_recommendation', 'retrieve_knowledge'],
}

def decide_agent_tools(event_type: str) -> list[str]:
    return EVENT_TOOL_PLAN.get(event_type, ['retrieve_knowledge'])

In [ ]:
APP_URL = 'https://your-vibefit-app.example'

def generate_personalized_email(event_type: str, user_id: str, display_name: str = 'صديق VibeFit') -> ProactiveEmail:
    tools = decide_agent_tools(event_type)
    context: dict[str, Any] = {'display_name': display_name}
    rag_docs = []
    used_personal = False
    used_rag = False

    if 'get_user_assessment' in tools:
        context['assessment'] = get_user_assessment(user_id)
        used_personal = context['assessment'] is not None
    if 'get_latest_recommendation' in tools:
        context['recommendation'] = get_latest_recommendation(user_id)
        used_personal = used_personal or context.get('recommendation') is not None
    if 'get_weekly_checkins' in tools:
        checkins = get_weekly_checkins(user_id)
        context['checkins'] = checkins
        if 'calculate_progress_summary' in tools:
            context['progress'] = calculate_progress_summary(checkins)
        used_personal = used_personal or len(checkins) > 0
    if 'retrieve_knowledge' in tools:
        query = 'التزام تعافي طاقة' if event_type == 'low_energy_detected' else 'التزام استمرارية'
        rag_docs = retrieve_knowledge(query)
        used_rag = len(rag_docs) > 0

    system = (
        'اكتب رسالة بريد عربية قصيرة وغير حكمية لـ VibeFit. '
        'لا Markdown. لا تشخيص طبي. 2-4 recommended_actions.'
    )
    user_prompt = {
        'event_type': event_type,
        'context': context,
        'rag': [{'title': d.get('title'), 'content': (d.get('content') or '')[:180]} for d in rag_docs[:3]],
        'cta_url': f'{APP_URL}/#/dashboard',
    }

    completion = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.5,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': json.dumps(user_prompt, ensure_ascii=False)},
        ],
    )
    raw = json.loads(completion.choices[0].message.content)
    email = ProactiveEmail(
        event_type=event_type,
        subject=raw.get('subject', 'رسالة من VibeFit'),
        greeting=raw.get('greeting', f'أهلًا {display_name}،'),
        body=raw.get('body', ''),
        recommended_actions=raw.get('recommended_actions', []),
        cta_text=raw.get('cta_text', 'افتحي VibeFit'),
        cta_url=raw.get('cta_url', f'{APP_URL}/#/dashboard'),
        footer_notice=raw.get('footer_notice', 'هذه توصيات عامة وليست تشخيصًا طبيًا.'),
        used_personal_data=used_personal,
        used_rag=used_rag,
        selected_tools=tools,
    )
    return email

In [ ]:
def demo(event_type: str, user_id: str, display_name: str = 'مروج'):
    email = generate_personalized_email(event_type, user_id, display_name)
    print('selected_tools:', email.selected_tools)
    print('used_rag:', email.used_rag)
    print('used_personal_data:', email.used_personal_data)
    print('subject:', email.subject)
    print('body:', email.body)
    print('recommended_actions:', email.recommended_actions)
    return email

## أمثلة تشغيل

استبدل `USER_ID` بمعرف مستخدم حقيقي من Supabase.

In [ ]:
USER_ID = '00000000-0000-0000-0000-000000000000'  # replace in demo

# welcome email
# demo('user_signed_up', USER_ID)

# assessment completed
# demo('assessment_completed', USER_ID)

# adherence dropped
# demo('adherence_dropped', USER_ID)

# low energy
# demo('low_energy_detected', USER_ID)

# weekly summary
# demo('weekly_summary_due', USER_ID)

print('Uncomment one demo() call after setting USER_ID')